<a href="https://colab.research.google.com/github/NamithaDaney/DataLens-AI/blob/main/DataLens_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas openpyxl

In [35]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI")

INPUT_DIR = BASE_DIR / "input"
REPORT_DIR = BASE_DIR / "reports"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_DIR = REPORT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Reports folder:", REPORT_DIR)
print("Plots folder:", PLOTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/input
Reports folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports
Plots folder: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports/plots


In [83]:
import time

SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xls"}
processed_files = set()

print("DataLens AI is watching:", INPUT_DIR)
print("Add a dataset to the input folder to start processing.")
print("Stop this cell using the Colab stop button.")

while True:
    for file_path in INPUT_DIR.iterdir():

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        # Process each filename only once per running session
        if file_path.name in processed_files:
            continue

        try:
            # Wait until the file size is stable before reading it
            size_before = file_path.stat().st_size
            time.sleep(2)
            size_after = file_path.stat().st_size

            if size_before != size_after or size_after == 0:
                continue

            process_dataset(file_path)
            processed_files.add(file_path.name)

        except Exception as error:
            print(f"Could not process {file_path.name}: {error}")

    time.sleep(10)


DataLens AI is watching: /content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/input
Add a dataset to the input folder to start processing.
Stop this cell using the Colab stop button.

Processing: cardhekho.xlsx
Possible index columns detected: ['Unnamed: 0']
Review these columns before deciding whether to remove them.
EDA visualization generation complete.
Number of plots generated: 8
/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports/plots/cardhekho/missing_values.png
/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports/plots/cardhekho/Unnamed_0_distribution.png
/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports/plots/cardhekho/ownerNo_distribution.png
/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_AI/reports/plots/cardhekho/modelYear_distribution.png
/content/drive/MyDrive/ENTRI Folder/ASSIGNMENTS/Final Main Project/DataLens_

KeyboardInterrupt: 

In [6]:
from pathlib import Path
import pandas as pd

def load_dataset(file_path):
    """
    Load a CSV or Excel file into a Pandas DataFrame.
    Detect possible index columns without automatically deleting them.
    Supports .csv, .xlsx and .xls files.
    """

    file_path = Path(file_path)
    extension = file_path.suffix.lower()

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if extension == ".csv":
        df = pd.read_csv(file_path,index_col=0)

    elif extension in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Please upload a CSV or Excel file."
        )

    if df.empty:
        raise ValueError("The uploaded file contains no data.")

    # Detect possible index columns
    unnamed_columns = [
        col for col in df.columns
        if str(col).strip().lower().startswith("unnamed:")
    ]

    if unnamed_columns:
        print("Possible index columns detected:", unnamed_columns)
        print("Review these columns before deciding whether to remove them.")

    return df


In [7]:
def review_unnamed_columns(df):
    """
    Detect possible index columns and recommend an action.
    Does not delete any columns automatically.
    """

    review_results = []

    for col in df.columns:
        column_name = str(col).strip().lower()

        if not column_name.startswith("unnamed:"):
            continue

        values = df[col]

        # Ignore missing values while checking the pattern
        non_missing = values.dropna()

        # Check whether values match 0, 1, 2, 3, ...
        numeric_values = pd.to_numeric(
            non_missing, errors="coerce"
        )

        looks_like_index = (
            len(non_missing) == len(df)
            and len(non_missing) > 0
            and numeric_values.notna().all()
            and numeric_values.reset_index(drop=True).equals(
                pd.Series(range(len(df)), dtype=numeric_values.dtype)
            )
        )

        if looks_like_index:
            recommendation = "Likely exported index — review for removal"
        else:
            recommendation = "Needs manual review — keep for now"

        review_results.append({
            "column": col,
            "missing_values": int(values.isna().sum()),
            "unique_values": int(values.nunique()),
            "sample_values": values.head(5).tolist(),
            "recommendation": recommendation
        })

    return pd.DataFrame(review_results)


In [8]:
def profile_dataset(df):
    """Generate a basic profile for any tabular dataset."""

    profile = {
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": df.columns.tolist(),
        "data_types": {
            col: str(df[col].dtype)
            for col in df.columns
        },
        "missing_values": {
            col: int(df[col].isna().sum())
            for col in df.columns
            if df[col].isna().sum() > 0
        },
        "missing_percentage": {
            col: round(
                df[col].isna().mean() * 100, 2
            )
            for col in df.columns
            if df[col].isna().sum() > 0
        },
        "duplicate_rows": int(df.duplicated().sum()),
        "unique_values": {
            col: int(df[col].nunique())
            for col in df.columns
        }
    }

    return profile


In [9]:
from itertools import combinations

def find_identical_columns(df):
    """Find exact duplicate columns without changing the dataset."""

    findings = []
    candidate_groups = {}

    # 1. Create a fingerprint for each column
    for position in range(df.shape[1]):
        series = df.iloc[:, position]

        fingerprint = pd.util.hash_pandas_object(
            series, index=False
        ).values.tobytes()

        candidate_groups.setdefault(
            fingerprint, []
        ).append(position)

    # 2. Compare only columns with matching fingerprints
    for positions in candidate_groups.values():
        if len(positions) < 2:
            continue

        for i, j in combinations(positions, 2):
            first = df.iloc[:, i]
            second = df.iloc[:, j]

            # Verify actual equality; never rely on hashes alone
            if first.equals(second):
                findings.append({
                    "column": str(df.columns[i]),
                    "issue": "Identical columns",
                    "count": len(df),
                    "severity": "Medium",
                    "recommendation": (
                        f"Column at position {j} "
                        f"('{df.columns[j]}') is identical to "
                        f"column at position {i} "
                        f"('{df.columns[i]}'). Review before "
                        "removing either column."
                    )
                })

    return findings


In [10]:
import pandas as pd
from itertools import combinations


def find_near_duplicate_columns(df):
    """
    Find text columns that become identical after removing
    leading/trailing spaces and ignoring letter case.
    Does not modify the original DataFrame.
    """
    findings = []
    candidate_groups = {}

    # 1. Normalize text columns only
    for position in range(df.shape[1]):
        series = df.iloc[:, position]

        if not (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        ):
            continue

        normalized = series.map(
            lambda value: (
                value.strip().casefold()
                if isinstance(value, str)
                else value
            )
        )

        # 2. Group potential matches efficiently
        fingerprint = pd.util.hash_pandas_object(
            normalized, index=False
        ).values.tobytes()

        candidate_groups.setdefault(fingerprint, []).append(
            (position, normalized)
        )

    # 3. Verify actual normalized equality
    for group in candidate_groups.values():
        if len(group) < 2:
            continue

        for (i, first), (j, second) in combinations(group, 2):
            if first.equals(second):
                findings.append({
                    "column": str(df.columns[i]),
                    "issue": "Potentially equivalent columns",
                    "count": len(df),
                    "severity": "Low",
                    "recommendation": (
                        f"Column at position {j} "
                        f"('{df.columns[j]}') matches after "
                        "normalizing text case and surrounding "
                        "spaces. Review before merging or removing."
                    )
                })

    return findings

In [23]:
import re
import pandas as pd

def find_inconsistent_date_formats(df):
    """Flag text columns containing multiple recognizable date formats."""

    findings = []

    # Recognize common date-shaped strings before parsing them.
    date_patterns = {
        "year-first": re.compile(
            r"^\d{4}[-/.]\d{1,2}[-/.]\d{1,2}$"
        ),
        "numeric-date": re.compile(
            r"^\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}$"
        ),
        "month-name": re.compile(
            r"^(?:\d{1,2}\s+)?"
            r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|"
            r"Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
            r"Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|"
            r"Nov(?:ember)?|Dec(?:ember)?)"
            r"(?:\s+\d{1,2})?,?\s+\d{4}$",
            re.IGNORECASE
        )
    }

    for col in df.columns:
        series = df[col]

        if not (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        ):
            continue

        text_values = series.dropna().astype(str).str.strip()

        if len(text_values) < 2:
            continue

        # Classify only values that resemble common date formats.
        def classify_date(value):
            for format_name, pattern in date_patterns.items():
                if pattern.fullmatch(value):
                    return format_name
            return None

        formats = text_values.map(classify_date)
        date_candidates = text_values[formats.notna()]
        candidate_formats = formats[formats.notna()]

        # Require at least two date-shaped values and two formats.
        if len(date_candidates) < 2:
            continue

        if candidate_formats.nunique() < 2:
            continue

        # Parse only date-shaped candidates, never arbitrary text.
        parsed = pd.to_datetime(
            date_candidates,
            format="mixed",
            errors="coerce"
        )

        valid = parsed.notna()

        if valid.sum() < 2:
            continue

        valid_formats = candidate_formats[valid]

        if valid_formats.nunique() < 2:
            continue

        # Count values using formats other than the most common format.
        format_counts = valid_formats.value_counts()
        inconsistent_count = int(
            len(valid_formats) - format_counts.iloc[0]
        )

        findings.append({
            "column": str(col),
            "issue": "Potentially inconsistent date formats",
            "count": inconsistent_count,
            "severity": "Low",
            "recommendation": (
                "Multiple date representations were detected. "
                "Review the intended format and ambiguous dates "
                "before standardizing."
            )
        })

    return findings

In [25]:
test_df = pd.DataFrame({
    "OrderDate": [
        "2025-01-15",
        "15/01/2025",
        "Jan 15, 2025",
        "2025-02-10"
    ],
    "Transmission": [
        "2.4 A/T",
        "5-speed M/T",
        "Automatic",
        "Manual"
    ],
    "Product": ["Pen", "Book", "Bag", "Pencil"]
})

test_result = pd.DataFrame(
    find_inconsistent_date_formats(test_df)
)

display(test_result)


,column,issue,count,severity,recommendation
0,OrderDate,Potentially inconsistent date formats,2,Low,Multiple date representations were detected. R...


In [22]:
import pandas as pd

def find_negative_numeric_values(df):
    """Flag negative values in numeric columns without modifying data."""

    findings = []

    for col in df.columns:
        series = df[col]

        # Check numeric columns only; exclude True/False values.
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
        ):
            continue

        negative_count = int((series < 0).sum())

        if negative_count > 0:
            findings.append({
                "column": str(col),
                "issue": "Negative numeric values",
                "count": negative_count,
                "severity": "Low",
                "recommendation": (
                    "Review negative values and confirm whether "
                    "they are valid for this column."
                )
            })

    return findings

In [80]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_numeric_outliers(df, file_path):
    """Detect IQR outliers and generate visualizations."""

    findings = []

    for col in df.columns:
        series = df[col]

        # Check numeric columns only; exclude Boolean columns.
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
        ):
            continue

        values = series.dropna()

        if len(values) < 4 or values.nunique() < 2:
            continue

        # Calculate IQR bounds.
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            continue

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (values < lower_bound) |
            (values > upper_bound)
        )

        outliers = values[outlier_mask]

        if outliers.empty:
            continue

        # Store actual values in the report.
        outlier_values = outliers.tolist()

        findings.append({
            "column": str(col),
            "issue": "Potential numeric outliers (IQR)",
            "count": len(outlier_values),
            "severity": "Low",
            "outlier_values": str(outlier_values),
            "lower_bound": round(float(lower_bound), 4),
            "upper_bound": round(float(upper_bound), 4),
            "recommendation": (
                "Review the listed values and confirm whether "
                "they are valid observations. Outliers are not "
                "automatically removed."
            )
        })

        # Generate a box plot for this column.
        plt.figure(figsize=(9, 4))
        plt.boxplot(values, vert=False)

        # Highlight the detected outliers.
        plt.scatter(
            outlier_values,
            [1] * len(outlier_values),
            color="red",
            label="Potential outliers",
            zorder=3
        )

        plt.axvline(
            lower_bound,
            linestyle="--",
            label="Lower IQR bound"
        )
        plt.axvline(
            upper_bound,
            linestyle="--",
            label="Upper IQR bound"
        )

        plt.title(f"Outlier Detection: {col}")
        plt.xlabel(str(col))
        plt.yticks([])
        plt.legend()
        plt.tight_layout()

        # Create a separate folder for this dataset's graphs
        dataset_plots_dir = PLOTS_DIR / file_path.stem
        dataset_plots_dir.mkdir(parents=True, exist_ok=True)

        # Create a safe filename for the column
        safe_col = re.sub(r"[^A-Za-z0-9_-]+", "_", str(col)).strip("_")

        # Save the outlier graph
        plot_file = dataset_plots_dir / f"{safe_col}_outlier_boxplot.png"
        plt.savefig(plot_file, dpi=150, bbox_inches="tight")

        #plt.show()
        plt.close()

    return findings


In [76]:
import math
import re
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def generate_eda_plots(df, file_path):
    """
    Generate general EDA visualizations for any DataFrame.
    Saves plots without modifying the original dataset.
    """

    # Create a separate plots folder for each dataset
    dataset_plots_dir = PLOTS_DIR / file_path.stem
    dataset_plots_dir.mkdir(parents=True, exist_ok=True)
    generated_plots = []

    # 1. Missing-values chart
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0].sort_values(
        ascending=False
    )

    if not missing_counts.empty:
        plt.figure(figsize=(12, 6))
        missing_counts.plot(kind="bar")
        plt.title("Missing Values by Column")
        plt.xlabel("Column")
        plt.ylabel("Missing Value Count")
        plt.xticks(rotation=75, ha="right")
        plt.tight_layout()

        plot_file = dataset_plots_dir / "missing_values.png"
        plt.savefig(plot_file, dpi=150, bbox_inches="tight")
        plt.close()
        generated_plots.append(str(plot_file))

    # 2. Numeric distribution histograms
    numeric_cols = [
        col for col in df.select_dtypes(include="number").columns
        if not pd.api.types.is_bool_dtype(df[col])
        and df[col].dropna().nunique() > 1
    ]

    for col in numeric_cols:
        values = df[col].dropna()

        if len(values) < 2:
            continue

        plt.figure(figsize=(8, 5))
        plt.hist(values, bins="auto", edgecolor="black")
        plt.title(f"Distribution of {col}")
        plt.xlabel(str(col))
        plt.ylabel("Frequency")
        plt.tight_layout()

        # Create a safe filename
        safe_col = re.sub(r"[^A-Za-z0-9_-]+", "_", str(col)).strip("_")
        plot_file = dataset_plots_dir / f"{safe_col}_distribution.png"

        plt.savefig(plot_file, dpi=150, bbox_inches="tight")
        plt.close()
        generated_plots.append(str(plot_file))

    # 3. Correlation heatmap
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr()

        if not corr.empty:
            plt.figure(figsize=(
                max(8, len(numeric_cols) * 0.7),
                max(6, len(numeric_cols) * 0.6)
            ))

            sns.heatmap(
                corr,
                cmap="coolwarm",
                center=0,
                annot=len(numeric_cols) <= 10,
                fmt=".2f"
            )

            plt.title("Numeric Feature Correlation Heatmap")
            plt.tight_layout()

            plot_file = dataset_plots_dir / "correlation_heatmap.png"
            plt.savefig(plot_file, dpi=150, bbox_inches="tight")
            plt.close()
            generated_plots.append(str(plot_file))

    print("EDA visualization generation complete.")
    print("Number of plots generated:", len(generated_plots))

    for plot in generated_plots:
        print(plot)

    return generated_plots


In [91]:
import pandas as pd

def run_data_quality_checks(df):
    """
    Identify common data-quality issues in any DataFrame.
    Does not modify or delete the original data.
    """

    results = []

    total_rows = len(df)

    for col in df.columns:
        series = df[col]
        missing_count = int(series.isna().sum())
        unique_count = int(series.nunique(dropna=True))

        # 1. Check missing values and completely empty columns
        if total_rows > 0 and missing_count == total_rows:
            results.append({
                "column": col,
                "issue": "Completely empty column",
                "count": missing_count,
                "severity": "High",
                "recommendation":
                    "All values in this column are NaN/null. "
                    "Consider deleting this column."
           })

        elif missing_count > 0:
            results.append({
                "column": col,
                "issue": "Missing values",
                "count": missing_count,
                "severity": (
                    "High" if missing_count / total_rows >= 0.5
                    else "Medium"
                ),
                "recommendation":
                    "Review missing values before deciding how to handle them."
            })

         # 2. Check constant columns
        if total_rows > 0 and unique_count == 1 :
            results.append({
                "column": col,
                "issue": "Constant",
                "count": unique_count,
                "severity": "Low",
                "recommendation":
                    "Check whether this column provides useful information."
            })

        # 3. Potential numeric data stored as text
        if pd.api.types.is_object_dtype(series) or \
           pd.api.types.is_string_dtype(series):

            non_missing = series.dropna()

            if len(non_missing) > 0:
                converted = pd.to_numeric(
                    non_missing.astype(str).str.replace(",", "", regex=False),
                    errors="coerce"
                )

                numeric_ratio = converted.notna().mean()

                if numeric_ratio >= 0.8:
                    results.append({
                        "column": col,
                        "issue": "Possible numeric data stored as text",
                        "count": int(converted.notna().sum()),
                        "severity": "Medium",
                        "recommendation":
                            "Review the values and convert to numeric if appropriate."
                    })

        # 4. High-cardinality columns
        if total_rows > 0 and unique_count / total_rows >= 0.9:
            results.append({
                "column": col,
                "issue": "High cardinality",
                "count": unique_count,
                "severity": "Low",
                "recommendation":
                    "Check whether this column contains IDs or mostly unique values."
            })

    # 5. Duplicate rows (dataset-level issue)
    duplicate_count = int(df.duplicated().sum())

    # 6. Identical columns
    results.extend(find_identical_columns(df))

    if duplicate_count > 0:
        results.append({
            "column": "(entire dataset)",
            "issue": "Duplicate rows",
            "count": duplicate_count,
            "severity": "Medium",
            "recommendation":
                "Review duplicate records before removing them."
        })

    # 7. Potentially equivalent columns
    results.extend(find_near_duplicate_columns(df))

    # 8. Potentially inconsistent date formats
    results.extend(find_inconsistent_date_formats(df))

    # 9. Potentially negative numeric values
    results.extend(find_negative_numeric_values(df))

    return pd.DataFrame(
        results,
        columns=[
            "column", "issue", "count",
            "severity", "recommendation"
        ]
    )


In [94]:
import json
import pandas as pd

def process_dataset(file_path):
    file_path = Path(file_path)

    print(f"\nProcessing: {file_path.name}")

    # 1. Load dataset
    df = load_dataset(file_path)

    # 2. Profile dataset
    profile = profile_dataset(df)

    # 3. Review suspicious Unnamed columns
    unnamed_report = review_unnamed_columns(df)

    # 4. Create missing-value report
    missing_report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (df.isna().mean() * 100).round(2)
    })

    missing_report = missing_report[
        missing_report["missing_count"] > 0
    ].sort_values("missing_percentage", ascending=False)

    # 5. Run data-quality checks
    quality_report = run_data_quality_checks(df)

   # 6. Generate the separate outlier report
    outlier_findings = find_numeric_outliers(df, file_path)

    outlier_report = pd.DataFrame(
       outlier_findings,
       columns=[
        "column",
        "issue",
        "count",
        "severity",
        "recommendation",
        "outlier_values",
        "lower_bound",
        "upper_bound"
      ]
    )


    # 7. Save reports using the dataset's filename
    report_name = file_path.stem

    with open(REPORT_DIR / f"{report_name}_profile.json", "w") as f:
        json.dump(profile, f, indent=4, default=str)

    missing_report.to_csv(
        REPORT_DIR / f"{report_name}_missing_values.csv"
    )

    unnamed_report.to_csv(
        REPORT_DIR / f"{report_name}_column_review.csv",
        index=False
    )

    quality_report.to_csv(
        REPORT_DIR / f"{report_name}_data_quality_report.csv",
        index=False
    )

    outlier_report.to_csv(
        REPORT_DIR / f"{report_name}_outlier_report.csv",
        index=False
    )

    eda_plots = generate_eda_plots(
    df, INPUT_DIR / "cardhekho.xlsx"
   )


    # 8. Print summary


    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Reports saved to:", REPORT_DIR)
    print(f"Outlier report saved: {outlier_file}")
    print(f"Number of outlier findings: {len(outlier_report)}")


    return df, profile, missing_report, unnamed_report
